In [4]:
import sys, os
sys.path.append(os.path.abspath(".."))
from src.bq_client import run

funnel_by_device = run(open("../sql/02_funnel_by_device.sql").read())
funnel_by_device

C:\Users\incha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,device,event_name,users
0,desktop,view_item,36323
1,desktop,add_to_cart,7384
2,desktop,begin_checkout,5665
3,desktop,add_payment_info,3352
4,desktop,purchase,2541
5,mobile,view_item,24810
6,mobile,add_to_cart,5142
7,mobile,begin_checkout,3968
8,mobile,add_payment_info,2387
9,mobile,purchase,1851


In [5]:
order = ['view_item', 'add_to_cart','begin_checkout','add_payment_info','purchase']

pivot = funnel_by_device.pivot(index='device', columns = 'event_name', values = 'users')[order]

step_rates = (pivot / pivot.shift(axis=1) * 100).round(1)

overall = (pivot['purchase']/pivot['view_item']*100).round(1)

print(pivot, "\n")
print(step_rates, "\n")
print(overall)

event_name  view_item  add_to_cart  begin_checkout  add_payment_info  purchase
device                                                                        
desktop         36323         7384            5665              3352      2541
mobile          24810         5142            3968              2387      1851
tablet           1443          276             219               134        97 

event_name  view_item  add_to_cart  begin_checkout  add_payment_info  purchase
device                                                                        
desktop          <NA>         20.3            76.7              59.2      75.8
mobile           <NA>         20.7            77.2              60.2      77.5
tablet           <NA>         19.1            79.3              61.2      72.4 

device
desktop    7.0
mobile     7.5
tablet     6.7
dtype: Float64


In [6]:
os.makedirs("../dashboard/data", exist_ok = True)

stage_order ={
    'view_item':1, 'add_to_cart':2, 'begin_checkout': 3,
    'add_payment_info':4, 'purchase':5
}

funnel_export = funnel_by_device.copy()
funnel_export['stage_order'] = funnel_export['event_name'].map(stage_order)
funnel_export.to_csv("../dashboard/data/funnel_by_device.csv",index=False)

funnel_export.head()

,device,event_name,users,stage_order
0,desktop,view_item,36323,1
1,desktop,add_to_cart,7384,2
2,desktop,begin_checkout,5665,3
3,desktop,add_payment_info,3352,4
4,desktop,purchase,2541,5
